[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Why Peewee &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the two models its worked examples wrote and the
catalog loaded into a database in memory. Run it first. The tasks only read, so they can be run in
any order.


In [1]:
import re
import sqlite3
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import CharField, DateField, ForeignKeyField, IntegerField, Model, SqliteDatabase

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

def sql(query):
    """The SQL a query will send, and the values that go with it, on one line."""
    statement, values = query.sql()
    return " ".join(statement.split()) + (f"  {values}" if values else "")

def message(error):
    """An error's class and text, without the memory address that makes no two runs agree."""
    return f"{type(error).__module__}.{type(error).__name__}: {re.sub(r'0x[0-9a-f]+', '0x...', str(error))}"

print("peewee", peewee.__version__, "| sqlite", sqlite3.sqlite_version)
print(len(AUTHORS), "authors and", len(BOOKS), "books in the catalog")


db = SqliteDatabase(":memory:")


class Author(Model):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()

    class Meta:
        database = db


class Book(Model):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()

    class Meta:
        database = db


db.create_tables([Author, Book])
for name, year in AUTHORS:
    Author.create(name=name, first_book=year)
written = {author.name: author for author in Author.select()}
for title, author, year, pages in BOOKS:
    Book.create(title=title, author=written[author], year=year, pages=pages)

print("loaded:", Author.select().count(), "authors and", Book.select().count(), "books")


peewee 4.5.1 | sqlite 3.50.4
4 authors and 12 books in the catalog
loaded: 4 authors and 12 books


**1.** The books from 1998 on, and the query that finds them.


In [2]:
recent = Book.select().where(Book.year >= 1998).order_by(Book.year)
print(sql(recent))
print([book.title for book in recent][:4], "...", recent.count(), "in all")


SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" WHERE ("t1"."year" >= ?) ORDER BY "t1"."year"  [1998]
['A Careful Fire', 'The Long Field', 'Stone and Tide', 'Winter Harbour'] ... 12 in all


The 1998 is a value beside the SQL rather than text inside it, which is what the `?` in the printed
query is.


**2.** How many books each author has.


In [3]:
for author in Author.select().order_by(Author.name):
    print(f"  {author.name:<15} {len(author.books)}")


  Ines O'Brien    3
  Kofi Mensah     3
  Marco Pietra    3
  Ursula Vance    3


`author.books` is the list the `backref` made, and reading it sends a query for each author, which
is the **Relationships** notebook's whole subject.


**3.** The same answer, by hand.


In [4]:
by_hand = sqlite3.connect(":memory:")
by_hand.execute("CREATE TABLE book (id INTEGER PRIMARY KEY, title TEXT, year INTEGER)")
by_hand.executemany("INSERT INTO book (title, year) VALUES (?, ?)",
                    [(title, year) for title, _, year, _ in BOOKS])

rows = by_hand.execute("SELECT title FROM book WHERE year >= ?", (1998,)).fetchall()
print("by hand:", len(rows), "rows | peewee:", Book.select().where(Book.year >= 1998).count())


by hand: 12 rows | peewee: 12


The same twelve rows either way. What differs is everything around the query: the table written out
by hand, the tuples coming back, and the placeholder that has to be remembered every time.


**4.** A model of your own, as a table.


In [5]:
class Loan(Model):
    borrower = CharField(max_length=60)
    copies = IntegerField()
    taken_on = DateField(index=True)

    class Meta:
        database = db


print(Loan._schema._create_table().query()[0])
for index in Loan._schema._create_indexes():
    print(index.query()[0])
# index=True on taken_on made the CREATE INDEX: it is a statement of its own, not part of the table.


CREATE TABLE IF NOT EXISTS "loan" ("id" INTEGER NOT NULL PRIMARY KEY, "borrower" VARCHAR(60) NOT NULL, "copies" INTEGER NOT NULL, "taken_on" DATE NOT NULL)
CREATE INDEX IF NOT EXISTS "loan_taken_on" ON "loan" ("taken_on")


The index is a second statement, which is why it does not appear in the `CREATE TABLE` above it.


**5.** The longest book, sorted by the database.


In [6]:
longest = Book.select().order_by(Book.pages.desc()).first()
print(longest.title, "|", longest.pages, "pages |", longest.author.name)
print(sql(Book.select().order_by(Book.pages.desc()).limit(1)))


Stone and Tide | 501 pages | Marco Pietra
SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" ORDER BY "t1"."pages" DESC LIMIT ?  [1]


`order_by` with `first()` becomes a `LIMIT 1` in the SQL, so the database returns one row rather
than twelve for Python to sort.


**6.** Everything one author wrote.


In [7]:
def books_by(name):
    """The titles an author wrote, oldest first, and an empty list for a name nobody has."""
    return [book.title for book in
            Book.select().join(Author).where(Author.name == name).order_by(Book.year)]


for name in ("Ines O'Brien", "Marco Pietra", "Nobody At All"):
    print(f"  {name:<15} {books_by(name)}")


  Ines O'Brien    ['A Careful Fire', 'The Long Field', 'Winter Harbour']
  Marco Pietra    ['Stone and Tide', 'The Lantern Keeper', 'Riverwork']
  Nobody At All   []


A name nobody has is an empty list rather than an error, since a `SELECT` that matches nothing is a
perfectly good `SELECT`. The apostrophe in the first name went into the query as a value, which is
where this notebook started.


---

&#8592; **Back to:** [Why Peewee](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/01-why-peewee.ipynb)  &nbsp;&middot;&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
